# Hypothesis Testing For Decision Making

**Official MA1001B Alignment:** *6.1 elements; 6.2 intervals and tests; 6.3 p-values; 6.4-6.7 tests for means, proportions, and variances.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Formulate null (`H0`) and alternative (`H1`) hypotheses for comparative A/B testing experiments.
- Execute two-sample contingency tests (`stats.chi2_contingency`) to evaluate conversion rate differences.
- Interpret p-values correctly as conditional probabilities under the null model, not as probabilities of the null itself.
- Differentiate between statistical significance ($p < 0.05$) and practical business significance (effect size lift).


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model experimental conversion differences under a null hypothesis of no effect to rigorously test design changes.
- **2. Computational Link (How Python represents it):** We use Pandas cross-tabulations (`pd.crosstab`) and SciPy chi-square tests (`stats.chi2_contingency`) to compute test statistics.
- **3. Decision Link (How it guides action):** Requiring both statistical significance and practical effect size lift prevents adopting costly modifications that yield trivial gains.


## Decision Scenario

> **The Problem:** A product owner wants to know whether a treatment page should replace a control page. The analysis must separate statistical evidence from practical business value.


## Conceptual Explanation

A hypothesis test asks whether the observed data would be surprising if a null claim were true. The p-value is not the probability that the null is true. It is a probability of data at least as extreme as what was observed, calculated under the null model.


## Mathematical Anchor

For two proportions, the practical effect is p_treatment - p_control. A test can evaluate evidence against equal conversion rates, but the decision should also consider effect size.


## Data And Workflow Notes

Uses a simulated A/B test unless Kaggle A/B testing files are later connected.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: A/B Test Simulation & Conversion Summary

We simulate an A/B test with 1,000 users in Control (10% baseline conversion) and 1,000 users in Treatment (12.5% conversion), then compute group conversion rates.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate A/B test records (n=1000 per group)
ab = pd.DataFrame({
    "group": np.repeat(["control", "treatment"], 1000),
    "converted": np.r_[rng.binomial(n=1, p=0.10, size=1000), rng.binomial(n=1, p=0.125, size=1000)],
})

# Calculate conversion rates by group
conversion_rates = ab.groupby("group")["converted"].agg(total_users="count", conversions="sum", rate="mean")
conversion_rates.round(4)


### Step 2: Contingency Table & Chi-Square Test of Independence

We construct a 2x2 contingency table of group versus conversion status and execute a Chi-Square test of independence to generate our test statistic and p-value.


In [ ]:
# Create 2x2 contingency table and run Chi-Square test
table = pd.crosstab(ab["group"], ab["converted"])
chi2_stat, p_value, dof, expected_counts = stats.chi2_contingency(table)

effect_lift = conversion_rates.loc["treatment", "rate"] - conversion_rates.loc["control", "rate"]
pd.Series({
    "control_rate": conversion_rates.loc["control", "rate"],
    "treatment_rate": conversion_rates.loc["treatment", "rate"],
    "absolute_conversion_lift": effect_lift,
    "chi_square_statistic": chi2_stat,
    "p_value": p_value,
}).round(4)


### Step 3: Bootstrap Confidence Interval for Effect Lift

To quantify uncertainty around the conversion lift, we generate 1,000 bootstrap resamples of both control and treatment groups and compute the 95% confidence interval of the difference.


In [ ]:
# Bootstrap 95% confidence interval for absolute conversion lift (Treatment - Control)
control_data = ab.loc[ab["group"].eq("control"), "converted"]
treatment_data = ab.loc[ab["group"].eq("treatment"), "converted"]

bootstrap_lifts = []
for seed in range(1000):
    c_mean = control_data.sample(n=len(control_data), replace=True, random_state=seed).mean()
    t_mean = treatment_data.sample(n=len(treatment_data), replace=True, random_state=seed + 10_000).mean()
    bootstrap_lifts.append(t_mean - c_mean)

pd.Series(bootstrap_lifts, name="bootstrap_lift_CI").quantile([0.025, 0.50, 0.975]).round(4)


### Step 4: Statistical Significance vs. Practical Importance

We evaluate our experimental results against a predefined business threshold (minimum practical lift of 2.0 percentage points) to make a definitive launch recommendation.


In [ ]:
# Evaluate decision against both statistical and practical thresholds
min_practical_lift = 0.020  # Business requires at least +2.0% lift to justify redesign costs

pd.Series({
    "observed_absolute_lift": effect_lift,
    "is_statistically_significant_(p<0.05)": p_value < 0.05,
    "meets_practical_business_threshold_(>=2.0%)": effect_lift >= min_practical_lift,
    "final_recommendation": "LAUNCH TREATMENT" if (p_value < 0.05 and effect_lift >= min_practical_lift) else "DO NOT LAUNCH"
})


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Would you recommend launching the treatment page if the p-value was 0.01 (statistically significant) but the observed conversion lift was only 0.2 percentage points? Justify your answer.

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Interpreting $p < 0.05$ as definitive proof that an experimental intervention has high practical importance.
- **Warning:** Ignoring effect size and confidence intervals when evaluating hypothesis test outcomes.
- **Warning:** Changing the hypothesis or primary success metric after seeing the experimental results (p-hacking / HARKing).


## Independent Practice

> [!TIP]
> **Your Task:**
> Modify the simulation so `treatment` has conversion probability `0.108` (only a minor +0.8% lift) and sample size is `10,000` per group. Run the test and explain why a tiny effect can be highly statistically significant in large samples.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What is one sentence or interpretation that you should never write when explaining a p-value to a stakeholder?

*Write your brief conceptual reflection below:*
